In [1]:
## Importing libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# supress the warnings
import warnings
warnings.filterwarnings('ignore')

In [3]:
# load dataset

df = pd.read_csv("../data/raw/warehouse_orders_raw.csv")
df.head()

,order_id,order_date,shift,warehouse_zone,priority,equipment_type,day_of_week,number_of_items,total_quantity,number_of_skus,order_weight_kg,picking_distance_m,picker_experience_months,number_of_pickers,warehouse_congestion,fragile_items,temperature_c,processing_time_minutes
0,104680,2026-07-14 23:00:00,Afternoon,B,Standard,Trolley,Tuesday,10,22,8,5.98,388.0,31.0,1,0.414,0,27.4,38.18
1,110474,2027-03-13 09:00:00,Morning,C,Standard,Manual,Saturday,11,33,7,18.97,507.2,31.0,1,0.260,0,25.5,60.78
2,110501,2027-03-14 12:00:00,Morning,B,Standard,Trolley,Sunday,9,28,8,5.91,403.5,8.0,1,0.254,0,26.7,36.50
3,108702,2026-12-29 13:00:00,Night,B,Standard,Manual,Tuesday,9,19,7,13.46,489.2,53.0,3,0.336,0,25.7,39.86
4,107174,2026-10-26 21:00:00,Afternoon,B,Standard,Manual,Monday,11,33,9,13.01,655.1,20.0,2,0.511,3,34.6,44.72


In [4]:
df.shape

(12025, 18)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12025 entries, 0 to 12024
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   order_id                  12025 non-null  int64  
 1   order_date                12025 non-null  object 
 2   shift                     12025 non-null  object 
 3   warehouse_zone            12025 non-null  object 
 4   priority                  11929 non-null  object 
 5   equipment_type            11904 non-null  object 
 6   day_of_week               12025 non-null  object 
 7   number_of_items           12025 non-null  int64  
 8   total_quantity            12025 non-null  int64  
 9   number_of_skus            12025 non-null  int64  
 10  order_weight_kg           11809 non-null  float64
 11  picking_distance_m        11881 non-null  float64
 12  picker_experience_months  11845 non-null  float64
 13  number_of_pickers         12025 non-null  int64  
 14  wareho

In [6]:
df['order_date'] = pd.to_datetime(df['order_date'])    #changing order date to Datetime format

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12025 entries, 0 to 12024
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   order_id                  12025 non-null  int64         
 1   order_date                12025 non-null  datetime64[ns]
 2   shift                     12025 non-null  object        
 3   warehouse_zone            12025 non-null  object        
 4   priority                  11929 non-null  object        
 5   equipment_type            11904 non-null  object        
 6   day_of_week               12025 non-null  object        
 7   number_of_items           12025 non-null  int64         
 8   total_quantity            12025 non-null  int64         
 9   number_of_skus            12025 non-null  int64         
 10  order_weight_kg           11809 non-null  float64       
 11  picking_distance_m        11881 non-null  float64       
 12  picker_experience_

In [17]:
##removing duplicates
df.duplicated().sum()

25

In [19]:
df = df.drop_duplicates().reset_index(drop=True)
df.shape

(12000, 18)

In [20]:
# separate target and feature variables

X = df.drop('processing_time_minutes', axis=1)
y = df['processing_time_minutes']

In [21]:
X.shape, y.shape

((12000, 17), (12000,))

In [12]:
type(X),type(y)

(pandas.core.frame.DataFrame, pandas.core.series.Series)

In [22]:
# Train Test Split (Split is done before teh imputations and preprocesing to avoid data leakage)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [23]:
X_train.shape, X_test.shape

((9600, 17), (2400, 17))

In [15]:
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns

print("Numerical columns:")
print(numeric_cols)

print("\nCategorical columns:")
print(categorical_cols)

Numerical columns:
Index(['order_id', 'number_of_items', 'total_quantity', 'number_of_skus',
       'order_weight_kg', 'picking_distance_m', 'picker_experience_months',
       'number_of_pickers', 'warehouse_congestion', 'fragile_items',
       'temperature_c'],
      dtype='object')

Categorical columns:
Index(['shift', 'warehouse_zone', 'priority', 'equipment_type', 'day_of_week'], dtype='object')


### Missing values treatment

In [24]:

print("Numerical missing values:")
print(X_train[numeric_cols].isnull().sum())

print("\nCategorical missing values:")
print(X_train[categorical_cols].isnull().sum())

Numerical missing values:
order_id                      0
number_of_items               0
total_quantity                0
number_of_skus                0
order_weight_kg             164
picking_distance_m          114
picker_experience_months    148
number_of_pickers             0
warehouse_congestion         98
fragile_items                 0
temperature_c                 0
dtype: int64

Categorical missing values:
shift              0
warehouse_zone     0
priority          80
equipment_type    87
day_of_week        0
dtype: int64


In [25]:
## Imputing Numerical columns with 'Median'.We have seen multiple outliers in the features, hence median is suitable imuputing value.

from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy='median')

X_train[numeric_cols] = num_imputer.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = num_imputer.transform(X_test[numeric_cols])

In [29]:
## Imputing categorical columns with 'Mode'

cat_imputer = SimpleImputer(strategy='most_frequent')

X_train[categorical_cols] = cat_imputer.fit_transform(X_train[categorical_cols])
X_test[categorical_cols] = cat_imputer.transform(X_test[categorical_cols])

In [31]:
# Rechecking MIssing values


print("Numerical missing values:")
print(X_train[numeric_cols].isnull().sum())

print("\nCategorical missing values:")
print(X_train[categorical_cols].isnull().sum())

Numerical missing values:
order_id                    0
number_of_items             0
total_quantity              0
number_of_skus              0
order_weight_kg             0
picking_distance_m          0
picker_experience_months    0
number_of_pickers           0
warehouse_congestion        0
fragile_items               0
temperature_c               0
dtype: int64

Categorical missing values:
shift             0
warehouse_zone    0
priority          0
equipment_type    0
day_of_week       0
dtype: int64


### Encoding categorical features 

In [33]:
from sklearn.preprocessing import OneHotEncoder    ##(We can use get_dummies method optionally)

encoder = OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False)

In [34]:
X_train_encoded = encoder.fit_transform(X_train[categorical_cols])

In [35]:
# transformating test data also

X_test_encoded = encoder.transform(X_test[categorical_cols])

In [36]:
encoder.get_feature_names_out(categorical_cols)

array(['shift_Morning', 'shift_Night', 'warehouse_zone_B',
       'warehouse_zone_C', 'warehouse_zone_D', 'warehouse_zone_E',
       'priority_Standard', 'priority_Urgent', 'equipment_type_Forklift',
       'equipment_type_Manual', 'equipment_type_Trolley',
       'day_of_week_Monday', 'day_of_week_Saturday', 'day_of_week_Sunday',
       'day_of_week_Thursday', 'day_of_week_Tuesday',
       'day_of_week_Wednesday'], dtype=object)

In [38]:
# converting to DataFrames

X_train_encoded_df = pd.DataFrame(
    X_train_encoded,
    columns=encoder.get_feature_names_out(categorical_cols),
    index=X_train.index
)

X_test_encoded_df = pd.DataFrame(
    X_test_encoded,
    columns=encoder.get_feature_names_out(categorical_cols),
    index=X_test.index
)

In [39]:
## concatenating the Econcoded columns with numerical columns

X_train_final = pd.concat([X_train[numeric_cols], X_train_encoded_df],axis=1)

X_test_final = pd.concat([X_test[numeric_cols], X_test_encoded_df],axis=1)

In [40]:
X_train_final.head()

,order_id,number_of_items,total_quantity,number_of_skus,order_weight_kg,picking_distance_m,picker_experience_months,number_of_pickers,warehouse_congestion,fragile_items,...,priority_Urgent,equipment_type_Forklift,equipment_type_Manual,equipment_type_Trolley,day_of_week_Monday,day_of_week_Saturday,day_of_week_Sunday,day_of_week_Thursday,day_of_week_Tuesday,day_of_week_Wednesday
9182,104742.0,13.0,41.0,9.0,27.530,585.0,14.0,2.0,0.352,3.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
11091,106494.0,10.0,50.0,8.0,14.875,437.8,14.0,2.0,0.321,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
6428,110281.0,10.0,32.0,9.0,12.160,476.0,33.0,2.0,0.497,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
288,103509.0,14.0,15.0,11.0,5.600,520.6,15.0,2.0,0.476,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
2626,105590.0,19.0,76.0,16.0,38.840,935.3,21.0,3.0,0.469,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [41]:
X_train_final.shape, X_test_final.shape

((9600, 28), (2400, 28))

In [43]:
X_train_final.columns

Index(['order_id', 'number_of_items', 'total_quantity', 'number_of_skus',
       'order_weight_kg', 'picking_distance_m', 'picker_experience_months',
       'number_of_pickers', 'warehouse_congestion', 'fragile_items',
       'temperature_c', 'shift_Morning', 'shift_Night', 'warehouse_zone_B',
       'warehouse_zone_C', 'warehouse_zone_D', 'warehouse_zone_E',
       'priority_Standard', 'priority_Urgent', 'equipment_type_Forklift',
       'equipment_type_Manual', 'equipment_type_Trolley', 'day_of_week_Monday',
       'day_of_week_Saturday', 'day_of_week_Sunday', 'day_of_week_Thursday',
       'day_of_week_Tuesday', 'day_of_week_Wednesday'],
      dtype='object')

In [44]:
X_test_final.columns

Index(['order_id', 'number_of_items', 'total_quantity', 'number_of_skus',
       'order_weight_kg', 'picking_distance_m', 'picker_experience_months',
       'number_of_pickers', 'warehouse_congestion', 'fragile_items',
       'temperature_c', 'shift_Morning', 'shift_Night', 'warehouse_zone_B',
       'warehouse_zone_C', 'warehouse_zone_D', 'warehouse_zone_E',
       'priority_Standard', 'priority_Urgent', 'equipment_type_Forklift',
       'equipment_type_Manual', 'equipment_type_Trolley', 'day_of_week_Monday',
       'day_of_week_Saturday', 'day_of_week_Sunday', 'day_of_week_Thursday',
       'day_of_week_Tuesday', 'day_of_week_Wednesday'],
      dtype='object')

In [49]:
X_train_final.isnull().sum()

order_id                    0
number_of_items             0
total_quantity              0
number_of_skus              0
order_weight_kg             0
picking_distance_m          0
picker_experience_months    0
number_of_pickers           0
warehouse_congestion        0
fragile_items               0
temperature_c               0
shift_Morning               0
shift_Night                 0
warehouse_zone_B            0
warehouse_zone_C            0
warehouse_zone_D            0
warehouse_zone_E            0
priority_Standard           0
priority_Urgent             0
equipment_type_Forklift     0
equipment_type_Manual       0
equipment_type_Trolley      0
day_of_week_Monday          0
day_of_week_Saturday        0
day_of_week_Sunday          0
day_of_week_Thursday        0
day_of_week_Tuesday         0
day_of_week_Wednesday       0
dtype: int64

In [50]:
X_test_final.isnull().sum()

order_id                    0
number_of_items             0
total_quantity              0
number_of_skus              0
order_weight_kg             0
picking_distance_m          0
picker_experience_months    0
number_of_pickers           0
warehouse_congestion        0
fragile_items               0
temperature_c               0
shift_Morning               0
shift_Night                 0
warehouse_zone_B            0
warehouse_zone_C            0
warehouse_zone_D            0
warehouse_zone_E            0
priority_Standard           0
priority_Urgent             0
equipment_type_Forklift     0
equipment_type_Manual       0
equipment_type_Trolley      0
day_of_week_Monday          0
day_of_week_Saturday        0
day_of_week_Sunday          0
day_of_week_Thursday        0
day_of_week_Tuesday         0
day_of_week_Wednesday       0
dtype: int64

In [51]:
X_train_final.dtypes

order_id                    float64
number_of_items             float64
total_quantity              float64
number_of_skus              float64
order_weight_kg             float64
picking_distance_m          float64
picker_experience_months    float64
number_of_pickers           float64
warehouse_congestion        float64
fragile_items               float64
temperature_c               float64
shift_Morning               float64
shift_Night                 float64
warehouse_zone_B            float64
warehouse_zone_C            float64
warehouse_zone_D            float64
warehouse_zone_E            float64
priority_Standard           float64
priority_Urgent             float64
equipment_type_Forklift     float64
equipment_type_Manual       float64
equipment_type_Trolley      float64
day_of_week_Monday          float64
day_of_week_Saturday        float64
day_of_week_Sunday          float64
day_of_week_Thursday        float64
day_of_week_Tuesday         float64
day_of_week_Wednesday       

In [52]:
X_test_final.dtypes

order_id                    float64
number_of_items             float64
total_quantity              float64
number_of_skus              float64
order_weight_kg             float64
picking_distance_m          float64
picker_experience_months    float64
number_of_pickers           float64
warehouse_congestion        float64
fragile_items               float64
temperature_c               float64
shift_Morning               float64
shift_Night                 float64
warehouse_zone_B            float64
warehouse_zone_C            float64
warehouse_zone_D            float64
warehouse_zone_E            float64
priority_Standard           float64
priority_Urgent             float64
equipment_type_Forklift     float64
equipment_type_Manual       float64
equipment_type_Trolley      float64
day_of_week_Monday          float64
day_of_week_Saturday        float64
day_of_week_Sunday          float64
day_of_week_Thursday        float64
day_of_week_Tuesday         float64
day_of_week_Wednesday       

In [ ]:
# before scaling all teh data is clean and do nto contrain any missing value


In [53]:
#dropping order_id column which is not required for training

X_train_final = X_train_final.drop('order_id', axis=1)
X_test_final = X_test_final.drop('order_id', axis=1)

In [54]:
X_train_final.shape, X_test_final.shape

((9600, 27), (2400, 27))

#### Scaling numeric columns

In [59]:
numeric_cols = numeric_cols.drop('order_id')

In [61]:
X_train_final[numeric_cols].head()

,number_of_items,total_quantity,number_of_skus,order_weight_kg,picking_distance_m,picker_experience_months,number_of_pickers,warehouse_congestion,fragile_items,temperature_c
9182,13.0,41.0,9.0,27.530,585.0,14.0,2.0,0.352,3.0,32.4
11091,10.0,50.0,8.0,14.875,437.8,14.0,2.0,0.321,1.0,21.0
6428,10.0,32.0,9.0,12.160,476.0,33.0,2.0,0.497,0.0,31.1
288,14.0,15.0,11.0,5.600,520.6,15.0,2.0,0.476,0.0,29.9
2626,19.0,76.0,16.0,38.840,935.3,21.0,3.0,0.469,1.0,25.8


In [62]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_final[numeric_cols] = scaler.fit_transform(X_train_final[numeric_cols])
X_test_final[numeric_cols] = scaler.transform(X_test_final[numeric_cols])

In [63]:
X_train_final[numeric_cols].head()

,number_of_items,total_quantity,number_of_skus,order_weight_kg,picking_distance_m,picker_experience_months,number_of_pickers,warehouse_congestion,fragile_items,temperature_c
9182,0.294153,0.475492,-0.290434,0.955817,0.005023,-0.819665,-0.202508,-0.716163,2.475334,1.367106
11091,-0.613843,0.994473,-0.570342,-0.215634,-0.806213,-0.819665,-0.202508,-0.983004,0.230875,-1.502142
6428,-0.613843,-0.043489,-0.290434,-0.466957,-0.595689,0.529285,-0.202508,0.531965,-0.891354,1.039911
288,0.596818,-1.023785,0.269382,-1.074205,-0.349893,-0.748667,-0.202508,0.351202,-0.891354,0.737885
2626,2.110144,2.493750,1.668923,2.002764,1.935567,-0.322683,0.750004,0.290947,0.230875,-0.294038


In [65]:
X_train_final[numeric_cols].describe()

,number_of_items,total_quantity,number_of_skus,order_weight_kg,picking_distance_m,picker_experience_months,number_of_pickers,warehouse_congestion,fragile_items,temperature_c
count,9.600000e+03,9.600000e+03,9.600000e+03,9.600000e+03,9.600000e+03,9.600000e+03,9.600000e+03,9.600000e+03,9.600000e+03,9.600000e+03
mean,2.168636e-16,-1.968795e-16,1.339669e-16,7.697546e-17,1.887379e-16,-1.276756e-16,2.479498e-16,-1.117625e-16,-1.199041e-16,6.439294e-17
std,1.000052e+00,1.000052e+00,1.000052e+00,1.000052e+00,1.000052e+00,1.000052e+00,1.000052e+00,1.000052e+00,1.000052e+00,1.000052e+00
min,-3.337829e+00,-1.715760e+00,-2.529698e+00,-1.555560e+00,-2.722979e+00,-1.742631e+00,-1.155021e+00,-2.317209e+00,-8.913541e-01,-3.641493e+00
25%,-6.138428e-01,-7.931273e-01,-5.703418e-01,-7.409593e-01,-6.598931e-01,-7.486674e-01,-1.155021e+00,-7.505936e-01,-8.913541e-01,-6.715702e-01
50%,-8.512457e-03,-1.588177e-01,-1.052571e-02,-2.156344e-01,-8.535927e-02,-1.806883e-01,-2.025081e-01,-6.197151e-02,2.308753e-01,7.988485e-03
75%,5.968178e-01,5.908210e-01,5.492904e-01,5.147292e-01,5.801080e-01,5.292855e-01,7.500044e-01,6.869050e-01,2.308753e-01,6.623783e-01
max,4.228800e+00,5.319311e+00,4.188095e+00,6.543237e+00,1.501730e+01,5.428105e+00,2.655030e+00,3.312277e+00,5.842022e+00,3.758146e+00


### saving the files

In [68]:
X_train_final.to_csv("X_train_final.csv", index=False)
X_test_final.to_csv("X_test_final.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)